In [1]:
import pandas as pd

In [2]:
caminho_arquivo = r'C:\Users\fabio\TCC\1_CFEM.csv'

In [3]:
# Defino o ponto de entrada para os dados brutos da CFEM. 
# Como se trata de um arquivo de fonte governamental, utilizo o encoding 'latin-1' 
# para garantir a leitura correta de caracteres especiais e acentuação.
caminho_arquivo = r'C:\Users\fabio\TCC\1_CFEM.csv'

try:
    # Realizo a leitura inicial do dataset mensal.
    df_cfem_mensal = pd.read_csv(
        caminho_arquivo,
        sep=',',
        encoding='latin-1'
    )
    
    # Diagnóstico Inicial: Verificação da estrutura e integridade dos dados.
    print("--- Visualização das Observações Iniciais ---")
    print(df_cfem_mensal.head())
    
    print("\n--- Inventário de Colunas Disponíveis ---")
    print(df_cfem_mensal.columns.tolist())
    
    print("\n--- Diagnóstico de Tipagem e Memória (Dtypes) ---")
    df_cfem_mensal.info()

except FileNotFoundError:
    print(f"ERRO CRÍTICO: O arquivo não foi localizado no diretório: {caminho_arquivo}")
except Exception as e:
    print(f"ERRO DE PROCESSAMENTO: Falha na leitura do dataset CFEM: {e}")

--- Visualização das Observações Iniciais ---
    Ano  Mês  Processo  AnoDoProcesso Tipo_PF_PJ        CPF_CNPJ Substância  \
0  2002    8  910262.0         2007.0         PJ  88503388000194    BASALTO   
1  2002    8  910262.0         2007.0         PJ  88503388000194    BASALTO   
2  2002    8  910262.0         2007.0         PJ  88503388000194    BASALTO   
3  2002    8  910262.0         2007.0         PJ  88503388000194    BASALTO   
4  2002    8  910262.0         2007.0         PJ  88503388000194    BASALTO   

   UF  CodigoMunicipio     Município QuantidadeComercializada UnidadeDeMedida  \
0  RS          4321808  TRÊS DE MAIO                        0             m3    
1  RS          4321808  TRÊS DE MAIO                        0             m3    
2  RS          4321808  TRÊS DE MAIO                        0             m3    
3  RS          4321808  TRÊS DE MAIO                        0             m3    
4  RS          4321808  TRÊS DE MAIO                        0             

In [4]:
# --- LIÇÃO 2: LIMPANDO A COLUNA 'ValorRecolhido' ---

print("Iniciando a normalização da variável 'ValorRecolhido'...")

# 1. LIMPEZA DE PREFIXOS E ESPAÇAMENTO
# Removo o símbolo monetário "R$" e espaços excedentes nas extremidades da string.
# O método .str.strip() garante que a base da string contenha apenas caracteres numéricos e separadores.
df_cfem_mensal['ValorRecolhido'] = df_cfem_mensal['ValorRecolhido'].str.strip('R$ ')

# 2. REMOÇÃO DE SEPARADORES DE MILHAR (PADRÃO ABNT)
# Elimino os pontos utilizados como separadores de milhar para evitar conflitos na conversão.
df_cfem_mensal['ValorRecolhido'] = df_cfem_mensal['ValorRecolhido'].str.replace('.', '', regex=False)

# 3. AJUSTE DO SEPARADOR DECIMAL
# Substituo a vírgula pelo ponto, adequando o dado ao padrão 'float' reconhecido pelo Python.
df_cfem_mensal['ValorRecolhido'] = df_cfem_mensal['ValorRecolhido'].str.replace(',', '.', regex=False)

# 4. CONVERSÃO DE TIPO (PARSING)
# Utilizo o pd.to_numeric para converter a série de strings tratadas em valores numéricos (float).
# Este passo é fundamental para permitir operações aritméticas e agregações estatísticas.
df_cfem_mensal['ValorRecolhido'] = pd.to_numeric(df_cfem_mensal['ValorRecolhido'])

# 5. VALIDAÇÃO PÓS-PROCESSAMENTO
# Verifico se a tipagem da coluna foi alterada com sucesso para float64.
print("\n--- Normalização Concluída: Diagnóstico de Tipagem Pós-Tratamento ---")
df_cfem_mensal.info()

Iniciando a normalização da variável 'ValorRecolhido'...

--- Normalização Concluída: Diagnóstico de Tipagem Pós-Tratamento ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2155944 entries, 0 to 2155943
Data columns (total 14 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   Ano                       int64  
 1   Mês                       int64  
 2   Processo                  float64
 3   AnoDoProcesso             float64
 4   Tipo_PF_PJ                object 
 5   CPF_CNPJ                  object 
 6   Substância                object 
 7   UF                        object 
 8   CodigoMunicipio           int64  
 9   Município                 object 
 10  QuantidadeComercializada  object 
 11  UnidadeDeMedida           object 
 12  ValorRecolhido            float64
 13  DataCriacao               object 
dtypes: float64(3), int64(3), object(8)
memory usage: 230.3+ MB


In [5]:
# --- LIÇÃO 3: CRIANDO A VERSÃO COMPACTA E ANUAL --- 

# Com os valores monetários devidamente normalizados, realizo a agregação 
# dos dados para converter a granularidade mensal em anual. Este procedimento 
# é essencial para alinhar a CFEM com as demais variáveis macroeconômicas do TCC.

print("Processando agrupamento de dados para consolidação anual...")

# 1. DEFINIÇÃO DAS CHAVES DE AGRUPAMENTO
# Seleciono as dimensões temporais e espaciais (Ano e Município) que servirão 
# como eixos para a consolidação dos valores.
chaves_agrupamento = ['Ano', 'CodigoMunicipio', 'Município']

# 2. CONSOLIDAÇÃO POR SOMA (GROUPBY)
# Utilizo o método .groupby() para somar o montante total recolhido 
# em cada unidade federativa municipal ao longo do ano fiscal.
df_compacto_anual = df_cfem_mensal.groupby(chaves_agrupamento)['ValorRecolhido'].sum()

# 3. NORMALIZAÇÃO DA ESTRUTURA (FLATTENING)
# Reseto o índice para retornar o objeto Series resultante a um DataFrame estruturado,
# facilitando futuras operações de merge e exportação.
df_compacto_anual = df_compacto_anual.reset_index()

# 4. INSPEÇÃO TÉCNICA DO DATASET ANUAL
print("--- Diagnóstico: Estrutura do Painel Anual ---")
print(df_compacto_anual.head())
print(df_compacto_anual.tail())

Processando agrupamento de dados para consolidação anual...
--- Diagnóstico: Estrutura do Painel Anual ---
    Ano  CodigoMunicipio              Município  ValorRecolhido
0  2002          4321808           TRÊS DE MAIO         2064.10
1  2003          1100015  ALTA FLORESTA D'OESTE          694.19
2  2003          1100023              ARIQUEMES         4603.68
3  2003          1100049                 CACOAL         7181.18
4  2003          1100098        ESPIGÃO D'OESTE         1169.77
        Ano  CodigoMunicipio       Município  ValorRecolhido
52531  2025          5221809          URUTAÍ         1355.44
52532  2025          5222005      VIANÓPOLIS         1915.58
52533  2025          5222054  VICENTINÓPOLIS          150.08
52534  2025          5222302   VILA PROPÍCIO      1539825.92
52535  2025          5300108        BRASÍLIA      8298739.42


In [6]:
# --- ETAPA 4: ESTRUTURAÇÃO DE PAINEL DETALHADO (AGREGAÇÃO POR SUBSTÂNCIA) ---

# Nesta etapa, realizo uma agregação mais granular para identificar a composição 
# da arrecadação da CFEM. Este detalhamento é fundamental para isolar o peso do 
# Nióbio na economia local em comparação a outras substâncias minerais.

print("Processando consolidação anual por categoria mineral...")

# 1. DEFINIÇÃO DAS CHAVES DE AGRUPAMENTO MULTINÍVEL
# Além das dimensões temporais e espaciais, incluo a variável 'Substância' 
# para desfragmentar o montante total arrecadado.
chaves_detalhadas = ['Ano', 'CodigoMunicipio', 'Município', 'Substância']

# 2. CONSOLIDAÇÃO SETORIAL (GROUPBY)
# Agrego os valores recolhidos garantindo que a soma respeite a tipologia mineral 
# de cada registro municipal.
df_detalhado_anual = df_cfem_mensal.groupby(chaves_detalhadas)['ValorRecolhido'].sum()

# 3. NORMALIZAÇÃO DA ESTRUTURA (RESET INDEX)
# Retorno os dados ao formato de DataFrame, facilitando filtros específicos 
# por tipo de minério durante a análise econométrica.
df_detalhado_anual = df_detalhado_anual.reset_index()

# 4. AUDITORIA DE ESTRUTURAS (DETALHADO VS. COMPACTO)
print("--- Diagnóstico: Painel Detalhado (Por Substância) ---")
print(df_detalhado_anual.head())

print("\n--- Diagnóstico: Painel Compacto (Total Municipal) ---")
print(df_compacto_anual.head())

Processando consolidação anual por categoria mineral...
--- Diagnóstico: Painel Detalhado (Por Substância) ---
    Ano  CodigoMunicipio              Município        Substância  \
0  2002          4321808           TRÊS DE MAIO           BASALTO   
1  2003          1100015  ALTA FLORESTA D'OESTE  BRITA DE GRANITO   
2  2003          1100023              ARIQUEMES       CASSITERITA   
3  2003          1100023              ARIQUEMES  GRANITO P/ BRITA   
4  2003          1100049                 CACOAL             AREIA   

   ValorRecolhido  
0         2064.10  
1          694.19  
2         3717.85  
3          885.83  
4         1929.55  

--- Diagnóstico: Painel Compacto (Total Municipal) ---
    Ano  CodigoMunicipio              Município  ValorRecolhido
0  2002          4321808           TRÊS DE MAIO         2064.10
1  2003          1100015  ALTA FLORESTA D'OESTE          694.19
2  2003          1100023              ARIQUEMES         4603.68
3  2003          1100049                 C

In [7]:
# --- ETAPA 5: EXPORTAÇÃO DOS DATASETS CONSOLIDADOS (OUTPUTS) ---

# Finalizo o módulo de tratamento da CFEM exportando os datasets anuais. 
# A separação em duas versões (Compacta e Detalhada) permite flexibilidade 
# tanto na modelagem do Controle Sintético quanto na análise de composição mineral.

# 1. EXPORTAÇÃO: DATASET COMPACTO (ARRECADAÇÃO TOTAL MUNICIPAL)
# Este arquivo será integrado à Base Mestra para as estimações de impacto.
caminho_saida_compacto = r'C:\Users\fabio\TCC\1.CFEM_FINAL.csv'
df_compacto_anual.to_csv(
    caminho_saida_compacto,
    index=False
)
print(f"Dataset COMPACTO (Total Municipal) exportado: {caminho_saida_compacto}")

# 2. EXPORTAÇÃO: DATASET DETALHADO (POR TIPO DE SUBSTÂNCIA)
# Este arquivo servirá para auditoria do peso do Nióbio e outros minérios na arrecadação.
caminho_saida_detalhado = r'C:\Users\fabio\TCC\VARIAVEIS\1.CFEM_FINAL_DETALHADO.csv'
df_detalhado_anual.to_csv(
    caminho_saida_detalhado,
    index=False
)
print(f"Dataset DETALHADO (Composição Mineral) exportado: {caminho_saida_detalhado}")

print("\n--- Módulo CFEM concluído com sucesso. ---")

Dataset COMPACTO (Total Municipal) exportado: C:\Users\fabio\TCC\1.CFEM_FINAL.csv
Dataset DETALHADO (Composição Mineral) exportado: C:\Users\fabio\TCC\VARIAVEIS\1.CFEM_FINAL_DETALHADO.csv

--- Módulo CFEM concluído com sucesso. ---
